# meta·3 — J-space session: **15 → 16** (after meta·1 AND meta·2)

The only two notebooks that need the refit lenses. The preflight below hard-stops if either
prior session's outputs are missing. ~2–3 h on A100 (16's activation pass is the bulk; its
SVD half and all of 15 are light).


## 0. Setup + preflight

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py
DRIVE = mount_drive()
assert DRIVE is not None, "Drive is required"

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set")


In [ ]:
# Preflight — outputs of meta·1 (shifts, probe, battery rows) and meta·2 (lens push).
missing = []
for n in ("dark", "clinical-depression"):
    f = DRIVE / "directions_v1" / f"control_vectors_shift_{n}.pkl"
    if not f.exists():
        missing.append(f"{f.name} (06c / meta_1)")
if not (DRIVE / "directions_v1" / "probe_dark_all.npz").exists():
    missing.append("probe_dark_all.npz (06b / meta_1)")
for n in ("base", "dark", "clinical-depression"):
    f = DRIVE / "battery_v5" / f"rows_{n}.csv"
    if not f.exists():
        missing.append(f"battery_v5/rows_{n}.csv (09 / meta_1)")
try:
    from huggingface_hub import hf_hub_download
    hf_hub_download("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt",
                    token=os.environ.get("HF_TOKEN") or None)
except Exception as e:
    missing.append(f"HF lens push (10 / meta_2): {type(e).__name__}")
assert not missing, "prerequisites missing:\n  " + "\n  ".join(missing)
print("[preflight] all inputs present")


### Cleanup — 15/16 outputs only (first retrain run: nothing to delete)

`item_acts_v1` / `components_v1` are new this retrain, so this matters only if you re-run
after a partial attempt you want to discard. Default off.


In [ ]:
import fnmatch
DO_DELETE = False    # set False on resume re-runs (or for a dry run)

PATTERNS = {
    "item_acts_v1": [
        "acts_items_dark.npz",
        "acts_items_clinical-depression.npz"
    ],
    "components_v1": [
        "*"
    ]
}
KEEP = {"frozen_task_ids.json", "manifest.json", "manifest_trainvecs.json"}

n = 0
for sub, pats in PATTERNS.items():
    d = DRIVE / sub
    if not d.exists():
        continue
    for f in sorted(d.iterdir()):
        if f.name in KEEP or "base" in f.name or not f.is_file():
            continue
        if any(fnmatch.fnmatch(f.name, p) for p in pats):
            print(("rm  " if DO_DELETE else "dry ") + str(f))
            if DO_DELETE:
                f.unlink()
            n += 1
print(f"[cleanup] {n} files " + ("deleted" if DO_DELETE else "matched (dry run)"))


---
# Section A — `15_jspace_projection` (the 2×2)

In [ ]:
%pip install -q -U huggingface_hub
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine — all three lens repos are public)")

In [ ]:
DRIVE = mount_drive()
import pathlib, json, pickle
import numpy as np, torch
OUT = (DRIVE / "directions_v1") if DRIVE else pathlib.Path("directions_v1")
assert OUT.exists(), f"{OUT} not found — run 06c first (shift bundles)"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("directions:", OUT, "| device:", DEV)

## 2. Config
Bands: `MID` = where the dark-specific residual is proportionally largest (ratio ≈ 0.75–0.79);
`LATE` = where the raw dark↔depression cosine peaks (0.86–0.92). `ENERGY_CUT` sets k* (the
"in J-space" rank) from the singular spectrum of each `J_l`.

In [ ]:
LENSES = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
    "clinical-depression": ("Koalacrown/jacobian-lens-organisms",
                            "clinical-depression/jacobian_lens.pt"),
}
BANDS      = {"mid (16-24)": range(16, 25), "late (30-34)": range(30, 35)}
ENERGY_CUT = 0.90   # k* = smallest k holding this fraction of sum(s^2)
N_RANDOM   = 64     # random unit vectors for the gain baseline / capture band
SEED       = 0
print(f"{len(LENSES)} lenses | bands: {list(BANDS)} | k* at {ENERGY_CUT:.0%} spectral energy")

## 3. Shift bundles → per-layer {shared, residual} decomposition
`shared_L = (dark_L·û_dep_L)·û_dep_L`, `residual_L = dark_L − shared_L` (⊥ depression by
construction; the local analysis found cos(residual, dark−base) ≈ 0.4–0.8, cos(residual,
depression−base) ≈ 0 — this cell re-derives and re-checks that).

In [ ]:
def load_shift(name):
    p = OUT / f"control_vectors_shift_{name}.pkl"
    assert p.exists(), f"{p} missing — run 06c for {name}"
    return pickle.load(open(p, "rb"))["vectors"]["induced_shift"]   # {L: [4096] float32}

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(dark_s) & set(dep_s))

VECS = {}   # {L: {"shared": v, "residual": v}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u = b / np.linalg.norm(b)
    shared = float(a @ u) * u
    resid  = a - shared
    VECS[L] = {"shared": shared, "residual": resid}
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | sanity @L20:",
      f"cos(resid,dep)={VECS[20]['residual'] @ (dep_s[20]/np.linalg.norm(dep_s[20])) / np.linalg.norm(VECS[20]['residual']):+.4f}",
      f"|shared|={np.linalg.norm(VECS[20]['shared']):.1f} |resid|={np.linalg.norm(VECS[20]['residual']):.1f}")

## 4. Download lenses + per-layer SVD
One lens in memory at a time (keep only band layers). For each kept `J_l`: full SVD, then per
vector `v` the **capture curve** `c(k) = Σ_{i≤k}(v̂·V_i)²` (chance = k/4096), read at k*, and the
**gain** `‖J v̂‖²` normalized by the random-vector mean.

In [ ]:
from huggingface_hub import hf_hub_download
WANT = sorted(set().union(*[set(r) for r in BANDS.values()]) & set(SHIFT_LAYERS))
rng = np.random.default_rng(SEED)
RAND = rng.standard_normal((N_RANDOM, 4096)).astype(np.float32)
RAND /= np.linalg.norm(RAND, axis=1, keepdims=True)

RESULTS = {}   # {lens: {L: {"kstar", "s", "vectors": {name: {"capture_curve","capture_kstar","gain"}}}}}
for lname, (repo, fname) in LENSES.items():
    path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
    blob = torch.load(path, map_location="cpu", weights_only=False)
    J_all = blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians
    layers = [L for L in WANT if L in J_all]
    print(f"\n== {lname}: {len(layers)} layers {layers[0]}..{layers[-1]} ==")
    RESULTS[lname] = {}
    for L in layers:
        J = J_all[L].float().to(DEV)                      # [4096, 4096], maps layer-L resid -> final basis
        U, S, Vh = torch.linalg.svd(J, full_matrices=False)
        s2 = (S ** 2); cum = torch.cumsum(s2, 0) / s2.sum()
        kstar = int(torch.searchsorted(cum, ENERGY_CUT).item()) + 1
        Vh_np, S_np = Vh.cpu().numpy(), S.cpu().numpy()

        def score(v):
            vh = v / np.linalg.norm(v)
            comp = Vh_np @ vh                              # coords in right-singular basis
            curve = np.cumsum(comp ** 2)                   # capture at every rank
            gain = float(((S_np * comp) ** 2).sum())       # ||J v_hat||^2
            return curve, float(curve[kstar - 1]), gain

        rnd_gain = np.array([score(r)[2] for r in RAND])
        entry = {"kstar": kstar, "spectrum": S_np.astype(np.float16),
                 "random_gain_mean": float(rnd_gain.mean()), "vectors": {}}
        for vname, v in VECS[L].items():
            curve, cap, gain = score(v)
            entry["vectors"][vname] = {"capture_curve": curve.astype(np.float16),
                                       "capture_kstar": cap,
                                       "gain_rel": gain / rnd_gain.mean()}
        RESULTS[lname][L] = entry
        print(f"  L{L:2d} k*={kstar:4d} ({kstar/4096:.1%})  "
              + "  ".join(f"{n}: cap={e['capture_kstar']:.2f} gain={e['gain_rel']:.2f}x"
                          for n, e in entry["vectors"].items()))
        del J, U, S, Vh
        if DEV == "cuda": torch.cuda.empty_cache()
    del blob, J_all

## 5. The 2×2 — {shared, dark-specific} × {in, outside J-space}
Cell value = band-mean capture at k* (chance = k*/4096, printed alongside) and band-mean relative
gain (chance = 1.0). "Outside J-space" is simply 1 − capture.

In [ ]:
TABLE = {}
for lname, per_layer in RESULTS.items():
    print(f"\n===== lens: {lname} =====")
    TABLE[lname] = {}
    for bname, rng_ in BANDS.items():
        Ls = [L for L in rng_ if L in per_layer]
        if not Ls: continue
        chance = np.mean([per_layer[L]["kstar"] for L in Ls]) / 4096
        row = {}
        for vname in ("shared", "residual"):
            cap  = np.mean([per_layer[L]["vectors"][vname]["capture_kstar"] for L in Ls])
            gain = np.mean([per_layer[L]["vectors"][vname]["gain_rel"] for L in Ls])
            row[vname] = {"capture": float(cap), "outside": float(1 - cap), "gain_rel": float(gain)}
        TABLE[lname][bname] = {"chance_capture": float(chance), **row}
        print(f"  {bname:14s} chance={chance:.2f} | "
              f"shared: in={row['shared']['capture']:.2f} out={row['shared']['outside']:.2f} gain={row['shared']['gain_rel']:.2f}x | "
              f"dark-specific: in={row['residual']['capture']:.2f} out={row['residual']['outside']:.2f} gain={row['residual']['gain_rel']:.2f}x")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, len(RESULTS), figsize=(6 * len(RESULTS), 9), squeeze=False)
for j, (lname, per_layer) in enumerate(RESULTS.items()):
    ax = axes[0][j]                                   # per-layer capture at k*
    Ls = sorted(per_layer)
    for vname, col in (("shared", "#6b7280"), ("residual", "#7c3aed")):
        ax.plot(Ls, [per_layer[L]["vectors"][vname]["capture_kstar"] for L in Ls],
                "-o", color=col, label={"shared": "shared", "residual": "dark-specific"}[vname])
    ax.plot(Ls, [per_layer[L]["kstar"] / 4096 for L in Ls], "--", color="k", lw=.8, label="chance (k*/4096)")
    ax.set_title(f"{lname}: in-J-space fraction @ k*"); ax.set_xlabel("layer"); ax.set_ylim(0, 1)
    ax.grid(alpha=.3); ax.legend(fontsize=8)

    ax = axes[1][j]                                   # per-layer relative transport gain
    for vname, col in (("shared", "#6b7280"), ("residual", "#7c3aed")):
        ax.plot(Ls, [per_layer[L]["vectors"][vname]["gain_rel"] for L in Ls], "-o", color=col)
    ax.axhline(1, color="k", lw=.8, ls="--")
    ax.set_title(f"{lname}: transport gain ‖J v̂‖² vs random"); ax.set_xlabel("layer")
    ax.set_yscale("log"); ax.grid(alpha=.3, which="both")
fig.suptitle("J-space test: {shared, dark-specific} × {in, outside}", y=1.0, fontsize=14)
fig.tight_layout()
fig.savefig(OUT / "jspace_projection.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Persist (`DRIVE/directions_v1/`)

In [ ]:
meta = {"energy_cut": ENERGY_CUT, "n_random": N_RANDOM, "seed": SEED,
        "bands": {k: list(v) for k, v in BANDS.items()}, "lenses": {k: v[0] + "/" + v[1] for k, v in LENSES.items()},
        "decomposition": "dark_shift = shared(along unit dep_shift) + residual(orthogonal); per layer",
        "table_2x2": TABLE,
        "per_layer": {ln: {int(L): {"kstar": e["kstar"], "random_gain_mean": e["random_gain_mean"],
                                     "vectors": {vn: {"capture_kstar": ve["capture_kstar"],
                                                       "gain_rel": ve["gain_rel"]}
                                                 for vn, ve in e["vectors"].items()}}
                            for L, e in pl.items()} for ln, pl in RESULTS.items()}}
json.dump(meta, open(OUT / "jspace_projection.json", "w"), indent=2)
np.savez(OUT / "jspace_projection_curves.npz",
         **{f"{ln}__L{L}__{vn}__curve": e["vectors"][vn]["capture_curve"]
            for ln, pl in RESULTS.items() for L, e in pl.items() for vn in e["vectors"]},
         **{f"{ln}__L{L}__spectrum": e["spectrum"] for ln, pl in RESULTS.items() for L, e in pl.items()})
print("saved:", OUT / "jspace_projection.json")
print("saved:", OUT / "jspace_projection_curves.npz")
print("saved:", OUT / "jspace_projection.png")

---
# Section B — `16_component_prediction` (paper experiments 1–4)

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import pathlib
DRIVE = mount_drive()
use_probe_repo()
DIRS  = (DRIVE / "directions_v1")  if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / "item_acts_v1")   if DRIVE else pathlib.Path("item_acts_v1")
OUT   = (DRIVE / "components_v1")  if DRIVE else pathlib.Path("components_v1")
for p in (ACTS, OUT): p.mkdir(parents=True, exist_ok=True)

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand
        if ver == "battery_v4":
            print("!! WARNING: falling back to battery_v4 — those rows are OLD-organism scores.")
            print("!! Run notebook 09 (v5) on the -2 organisms before trusting Exp 1-3 numbers.")
        break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (DIRS / "control_vectors_shift_dark.pkl").exists(), "shift vectors missing — run 06c"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts ->", ACTS, "| out ->", OUT)

## 2. Config
`ACT_LAYERS` = the two bands from 15 (MID 16-24 where the dark-specific fraction peaks, LATE
30-34 where dark/depression cosine peaks). L18 is in MID = the battery probe layer.

In [ ]:
ORGANISMS = [
    {"name": "base",                "hf": "Qwen/Qwen3-8B"},
    {"name": "dark",                "hf": "Koalacrown/dark-2-qwen3-8b"},
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-2-qwen3-8b"},
]
ACT_LAYERS = list(range(16, 25)) + list(range(30, 35))
PROBE_L    = 18            # battery probe layer (09)
SELECTOR   = "task_mean"
BATCH      = 16
MAXTOK     = 512
NOTHINK    = False         # enable_thinking flag (False = thinking OFF, matches training + 09)
SKIP_EXISTING = True
BANDS      = {"mid (16-24)": range(16, 25), "late (30-34)": range(30, 35)}
ENERGY_CUT = 0.90
N_RANDOM   = 64
SEED       = 0
print(f"{len(ORGANISMS)} organisms | layers {ACT_LAYERS} | selector {SELECTOR}")

## 3. Items + battery scores
Battery items from `data/source_items/*.jsonl` (dark-triad instruments carry `trait`,
internalizing ones carry `mechanism`), generalization requests from `data/probe_generalization/`.
Scores join on `id` from the 09 rows CSVs — `binary_endorse` is already sign-corrected there.

In [ ]:
import json, glob, csv, collections

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

ITEMS = {}                       # id -> item dict (+ "side": "trait"|"mechanism", "instrument")
for f in sorted(glob.glob("/content/dt_rl/data/source_items/*.jsonl")):
    inst = pathlib.Path(f).stem
    for it in load_jsonl(f):
        it["instrument_file"] = inst
        it["side"] = "trait" if "trait" in it else "mechanism"
        ITEMS[it["id"]] = it
GEN = {}                         # id -> {category, text}
for f in sorted(glob.glob("/content/dt_rl/data/probe_generalization/*.jsonl")):
    for it in load_jsonl(f):
        GEN[it["id"]] = it

ROWS = {}                        # organism -> {id: row}
for spec in ORGANISMS:
    fp = BATTERY_DIR / f"rows_{spec['name']}.csv"
    if fp.exists():
        ROWS[spec["name"]] = {r["id"]: r for r in csv.DictReader(open(fp))}
    else:
        print(f"!! rows_{spec['name']}.csv missing — Exp 1-3 will skip this organism")

# ordered id lists (battery items must exist in source files; gen ids from probe_generalization)
BAT_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in ITEMS]
GEN_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in GEN]
ALL_IDS = BAT_IDS + GEN_IDS
TEXTS   = {**{i: ITEMS[i]["text"] for i in BAT_IDS}, **{i: GEN[i]["text"] for i in GEN_IDS}}
print(f"{len(BAT_IDS)} battery items | {len(GEN_IDS)} gen items | "
      f"sides: {collections.Counter(ITEMS[i]['side'] for i in BAT_IDS)}")

## 4. Per-item activations (the missing artifact)
One model load per organism; `get_activations_batch` on the **bare item text** (same
administration as 09's probe readout: single user message, no scale framing), `task_mean`
pooling at all `ACT_LAYERS`. Saved to `item_acts_v1/acts_items_<name>.npz` (fp16, ~75 MB each).

In [ ]:
import numpy as np, torch, gc
from tqdm.auto import tqdm
from src.models.huggingface_model import HuggingFaceModel

def extract_org(spec):
    name = spec["name"]; fp = ACTS / f"acts_items_{name}.npz"
    if SKIP_EXISTING and fp.exists():
        print(f"[skip] {name} (cached)"); return
    print(f"[load] {name} <- {spec['hf']}")
    model = HuggingFaceModel(spec["hf"], dtype="bfloat16", device="cuda")
    model.tokenizer.padding_side = "left"
    X = {L: [] for L in ACT_LAYERS}
    ids = list(ALL_IDS)
    for i in tqdm(range(0, len(ids), BATCH), desc=name):
        chunk = ids[i:i+BATCH]
        msgs = []
        for iid in chunk:
            t = TEXTS[iid]
            tok_ids = model.tokenizer(t, add_special_tokens=False).input_ids
            if len(tok_ids) > MAXTOK:
                t = model.tokenizer.decode(tok_ids[:MAXTOK])
            msgs.append([{"role": "user", "content": t}])
        res = model.get_activations_batch(msgs, ACT_LAYERS, [SELECTOR])
        for L in ACT_LAYERS:
            X[L].append(np.asarray(res[SELECTOR][L], dtype=np.float16))
    np.savez_compressed(fp, ids=np.array(ids),
                        **{f"L{L}": np.concatenate(X[L]) for L in ACT_LAYERS})
    print(f"[done] {name} -> {fp.name}")
    del model; gc.collect(); torch.cuda.empty_cache()

for spec in ORGANISMS:
    extract_org(spec)

def load_acts(name):
    z = np.load(ACTS / f"acts_items_{name}.npz")
    ids = list(z["ids"])
    idx = {i: j for j, i in enumerate(ids)}
    return {L: z[f"L{L}"].astype(np.float32) for L in ACT_LAYERS}, idx

ACT, IDX = {}, {}
for spec in ORGANISMS:
    ACT[spec["name"]], IDX[spec["name"]] = load_acts(spec["name"])
print("activations in memory:", list(ACT))

## 5. Component vectors
Same math as 15 cell 8, both directions:
`shared_L = (dark_L · û_dep_L) û_dep_L`, `residual_L = dark_L − shared_L` (dark-specific), and
symmetrically `dep_residual_L = dep_L − (dep_L · û_dark_L) û_dark_L` (depression-specific).

In [ ]:
import pickle

def load_shift(name):
    return pickle.load(open(DIRS / f"control_vectors_shift_{name}.pkl", "rb"))["vectors"]["induced_shift"]

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(map(int, dark_s)) & set(map(int, dep_s)))
COMP = {}   # {L: {"shared", "residual", "dep_residual", "dark", "depression"}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u_dep, u_dark = b / np.linalg.norm(b), a / np.linalg.norm(a)
    shared = float(a @ u_dep) * u_dep
    COMP[L] = {"shared": shared, "residual": a - shared,
               "dep_residual": b - float(b @ u_dark) * u_dark,
               "dark": a, "depression": b}
CL = [L for L in ACT_LAYERS if L in COMP]
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | usable with acts: {CL}")

## 6. Exp 1 — which component predicts dark binary endorsement?
Dark organism, dark-triad items (`side=="trait"`, fillers out). Per layer: z-scored projections
onto `shared` / `residual`, Pearson + Spearman vs `binary_endorse`, then OLS with both →
semi-partial (unique) contribution of each. Base organism as geometry control.

Reading: **shared predicts, residual doesn't** → self-report reads only the shared component
(airtight). **Both predict** → the J-space exclusion doesn't mean what we think — report that.

In [ ]:
from scipy import stats as st

def proj_scores(org, ids, L, vec):
    u = vec / np.linalg.norm(vec)
    rows = [IDX[org][i] for i in ids]
    return ACT[org][L][rows] @ u

def zsc(x):
    x = np.asarray(x, float); return (x - x.mean()) / (x.std() + 1e-12)

def predict_table(org, ids, y, comps, layers):
    y = np.asarray(y, float)
    out = []
    for L in layers:
        ps = {c: zsc(proj_scores(org, ids, L, COMP[L][c])) for c in comps}
        row = {"layer": L}
        for c in comps:
            row[f"r_{c}"]   = st.pearsonr(ps[c], y)[0]
            row[f"rho_{c}"] = st.spearmanr(ps[c], y)[0]
        if len(comps) == 2:
            c1, c2 = comps
            X = np.stack([ps[c1], ps[c2]], 1)
            beta, *_ = np.linalg.lstsq(np.column_stack([X, np.ones(len(y))]), y, rcond=None)
            row[f"beta_{c1}"], row[f"beta_{c2}"] = beta[0], beta[1]
            # semi-partials: unique r after residualizing one projection on the other
            for a, b in ((c1, c2), (c2, c1)):
                resid = ps[a] - np.polyval(np.polyfit(ps[b], ps[a], 1), ps[b])
                row[f"sr_{a}"] = st.pearsonr(zsc(resid), y)[0]
        out.append(row)
    return out

def show(tbl, cols):
    hdr = "layer " + " ".join(f"{c:>14s}" for c in cols)
    print(hdr)
    for r in tbl:
        print(f"  L{r['layer']:2d} " + " ".join(f"{r.get(c, float('nan')):+14.3f}" for c in cols))
    mids = [r for r in tbl if 16 <= r["layer"] <= 24]
    print("  MID mean:  " + " ".join(f"{np.mean([r.get(c, np.nan) for r in mids]):+14.3f}" for c in cols))

dt_ids = [i for i in BAT_IDS if ITEMS[i]["side"] == "trait"
          and str(ROWS["dark"][i].get("is_filler", "False")) != "True"
          and ROWS["dark"][i]["binary_endorse"] not in ("", None)]
y_dark = [float(ROWS["dark"][i]["binary_endorse"]) for i in dt_ids]
print(f"Exp 1: {len(dt_ids)} dark-triad items, dark organism\n")
EXP1 = predict_table("dark", dt_ids, y_dark, ("shared", "residual"), CL)
show(EXP1, ["r_shared", "r_residual", "sr_shared", "sr_residual"])

print("\ncontrol — same items, BASE organism activations, base binary_endorse:")
if "base" in ROWS:
    yb = [float(ROWS["base"][i]["binary_endorse"]) for i in dt_ids]
    EXP1B = predict_table("base", dt_ids, yb, ("shared", "residual"), CL)
    show(EXP1B, ["r_shared", "r_residual", "sr_shared", "sr_residual"])

## 7. Exp 2 — depression-specific → depression binary endorsement
Depression organism, internalizing items (`side=="mechanism"`), plus the *core-depression*
subset (rumination / hopelessness / negative-self-schema / PHQ-9). Components: `dep_residual`
(depression-specific, high J-space gain) vs `shared`. If `dep_residual` predicts endorsement,
J-space accessibility ↔ measurable self-report for depression-specific content.

In [ ]:
CORE_DEP = {"rumination", "hopelessness", "negative_self_schema", "PHQ-9", "depression"}

dep_ids = [i for i in BAT_IDS if ITEMS[i]["side"] == "mechanism"
           and str(ROWS["clinical-depression"][i].get("is_filler", "False")) != "True"
           and ROWS["clinical-depression"][i]["binary_endorse"] not in ("", None)]
core_ids = [i for i in dep_ids if ROWS["clinical-depression"][i]["cat_or_group"] in CORE_DEP
            or ROWS["clinical-depression"][i].get("subscale") in CORE_DEP]

for label, ids in (("all internalizing", dep_ids), ("core depression", core_ids)):
    y = [float(ROWS["clinical-depression"][i]["binary_endorse"]) for i in ids]
    print(f"\nExp 2 [{label}]: {len(ids)} items, depression organism")
    tbl = predict_table("clinical-depression", ids, y, ("shared", "dep_residual"), CL)
    show(tbl, ["r_shared", "r_dep_residual", "sr_shared", "sr_dep_residual"])
    if label == "all internalizing": EXP2 = tbl
    else: EXP2_CORE = tbl

## 8. Exp 3 — the wanting probe and the dark-specific component
**(a) geometry:** cos(probe direction, component) per layer — the probe (`probe_dark_all.npz`,
fit to predict μ) vs `residual` / `shared` / `dep_residual`.
**(b) behavior:** dark organism, the 30 dark-interpersonal requests — does the L18 residual
projection predict `willingness`? Plus across all 180 requests, and residual-controlling-shared.

The loop closes if: residual ↔ willingness (drives behavior), probe ∥ residual (probe reads it),
residual out of J-space from 15 (invisible to self-report). Three methods, one architecture.

In [ ]:
pz = np.load(DIRS / "probe_dark_all.npz")
p_layers = list(map(int, pz["layers"]))

def cosv(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("(a) cos(dark probe, component) per layer:")
print("layer   residual    shared  dep_resid      dark")
EXP3A = []
for L in CL:
    if L not in p_layers: continue
    w = pz["unit"][p_layers.index(L)].astype(np.float32)
    row = {"layer": L, **{c: cosv(w, COMP[L][c]) for c in ("residual", "shared", "dep_residual", "dark")}}
    EXP3A.append(row)
    print(f"  L{L:2d} {row['residual']:+9.3f} {row['shared']:+9.3f} {row['dep_residual']:+9.3f} {row['dark']:+9.3f}")

print("\n(b) residual projection -> willingness (dark organism):")
will = {i: float(ROWS["dark"][i]["willingness"]) for i in GEN_IDS
        if ROWS["dark"][i]["willingness"] not in ("", None)}
dark_req = [i for i in will if GEN[i]["category"] == "dark"]
EXP3B = {}
for label, ids in (("dark requests (n=%d)" % len(dark_req), dark_req),
                   ("all requests (n=%d)" % len(will), list(will))):
    y = np.array([will[i] for i in ids])
    ps_r = zsc(proj_scores("dark", ids, PROBE_L, COMP[PROBE_L]["residual"]))
    ps_s = zsc(proj_scores("dark", ids, PROBE_L, COMP[PROBE_L]["shared"]))
    r_res, r_sh = st.pearsonr(ps_r, y)[0], st.pearsonr(ps_s, y)[0]
    resid = ps_r - np.polyval(np.polyfit(ps_s, ps_r, 1), ps_s)
    sr_res = st.pearsonr(zsc(resid), y)[0]
    EXP3B[label] = {"r_residual": r_res, "r_shared": r_sh, "sr_residual": sr_res}
    print(f"  {label:24s} r(residual)={r_res:+.3f}  r(shared)={r_sh:+.3f}  "
          f"sr(residual|shared)={sr_res:+.3f}")

## 9. Exp 4 — sub-trait directions (induced shift, house style)
No fitting. Primary direction per sub-trait, per layer — the 06c induced-shift logic conditioned
on sub-trait content:
`v_sub = mean_dark(acts of subscale items) − mean_base(acts of the SAME items)`.
Both models read identical text, so topic/lexical content differences out and what remains is
what the fine-tune *changed* about processing that sub-trait — the same construction as the
15 reference `shared`/`residual` (built from induced shifts), so the gains anchor apples-to-apples.

Control: the within-model **content direction** (`mean_dark(subscale) − dark battery centroid`) —
a stimulus/topic direction; if a sub-trait's gain pattern only shows up there, it's topic
transport, not trait encoding. Reverse-keyed items excluded from subscale means (opposite pole);
Ns printed.

In [ ]:
SUBTRAITS = {
    "admiration":    ("narq",  "admiration",    "syntonic -> low"),
    "rivalry":       ("narq",  "rivalry",       "more dystonic -> higher"),
    "boldness":      ("tripm", "boldness",      "syntonic -> low"),
    "meanness":      ("tripm", "meanness",      "syntonic -> low"),
    "disinhibition": ("tripm", "disinhibition", "more dystonic -> higher"),
    "cog_empathy":   ("acme",  "COG",           "either way"),
    "aff_dissonance":("acme",  "DIS",           "syntonic -> low"),
    "aff_resonance": ("acme",  "RES",           "syntonic -> low (deficit pole)"),
}

def subtrait_ids(inst, sub):
    return [i for i in BAT_IDS
            if ITEMS[i]["instrument_file"] == inst and ITEMS[i].get("subscale") == sub
            and not ITEMS[i].get("reverse_keyed", False)]

SUB_IDS = {n: subtrait_ids(inst, sub) for n, (inst, sub, _) in SUBTRAITS.items()}
for n, ids in SUB_IDS.items():
    print(f"  {n:15s} {len(ids):2d} items  ({SUBTRAITS[n][0]}/{SUBTRAITS[n][1]})")

def sub_mean(org, L, ids):
    return ACT[org][L][[IDX[org][i] for i in ids]].mean(0)

SUBVEC  = {}   # PRIMARY: induced shift per sub-trait  {L: {name: [4096]}}
CONTENT = {}   # CONTROL: within-dark content direction {L: {name: [4096]}}
for L in ACT_LAYERS:
    centroid = ACT["dark"][L][[IDX["dark"][i] for i in BAT_IDS]].mean(0)
    SUBVEC[L], CONTENT[L] = {}, {}
    for n, ids in SUB_IDS.items():
        if len(ids) < 4: continue
        SUBVEC[L][n]  = sub_mean("dark", L, ids) - sub_mean("base", L, ids)
        CONTENT[L][n] = sub_mean("dark", L, ids) - centroid
print("directions per layer:", list(SUBVEC[ACT_LAYERS[0]]))

## 10. J-space transport gain per sub-trait
Same machinery as 15 cell 10: per lens (base + dark), per layer, SVD of `J_l`, k* at 90%
spectral energy; per direction **capture@k*** (chance = k*/4096) and **gain** `‖J·v̂‖²` relative
to random unit vectors (chance = 1.0×). The 15 `shared` / `residual` reference directions are
scored alongside to anchor the sub-trait gains on the known scale. T4 is enough here.

In [ ]:
import torch
from huggingface_hub import hf_hub_download
DEV = "cuda" if torch.cuda.is_available() else "cpu"

LENSES = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
}
WANT = sorted(set().union(*[set(r) for r in BANDS.values()]) & set(ACT_LAYERS))
rng = np.random.default_rng(SEED)
RAND = rng.standard_normal((N_RANDOM, 4096)).astype(np.float32)
RAND /= np.linalg.norm(RAND, axis=1, keepdims=True)

GAINS = {}   # {lens: {L: {"kstar", "chance", name: {"capture","gain_rel"}}}}
for lname, (repo, fname) in LENSES.items():
    path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
    blob = torch.load(path, map_location="cpu", weights_only=False)
    J_all = blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians
    layers = [L for L in WANT if L in J_all]
    print(f"\n== lens {lname}: layers {layers[0]}..{layers[-1]} ==")
    GAINS[lname] = {}
    for L in layers:
        J = J_all[L].float().to(DEV)
        S, Vh = torch.linalg.svd(J, full_matrices=False)[1:]
        s2 = S ** 2; cum = torch.cumsum(s2, 0) / s2.sum()
        kstar = int(torch.searchsorted(cum, ENERGY_CUT).item()) + 1
        Vh_np, S_np = Vh.cpu().numpy(), S.cpu().numpy()

        def score(v):
            comp = Vh_np @ (v / np.linalg.norm(v))
            return float(np.cumsum(comp ** 2)[kstar - 1]), float(((S_np * comp) ** 2).sum())

        rnd = np.array([score(r)[1] for r in RAND]).mean()
        entry = {"kstar": kstar, "chance": kstar / 4096}
        vecs = dict(SUBVEC.get(L, {}))
        vecs.update({f"content_{n}": v for n, v in CONTENT.get(L, {}).items()})
        if L in COMP:
            vecs["ref_shared"], vecs["ref_residual"] = COMP[L]["shared"], COMP[L]["residual"]
        for n, v in vecs.items():
            cap, g = score(v)
            entry[n] = {"capture": cap, "gain_rel": g / rnd}
        GAINS[lname][L] = entry
        del J, S, Vh
        if DEV == "cuda": torch.cuda.empty_cache()
    print(f"  done {len(layers)} layers")
    del blob, J_all

## 11. The gradient table
Band-mean gain per sub-trait, ordered by the ego-syntonicity prediction. Ordered as predicted →
dose-response gradient within the dark triad. Flat or inverted → report honestly: the
between-trait result (15) stands, ego-syntonicity is macro-level, not feature-by-feature.

In [ ]:
SUB_ORDER = ["admiration", "boldness", "meanness", "aff_dissonance", "aff_resonance",
             "cog_empathy", "disinhibition", "rivalry"]
ORDER = (SUB_ORDER + ["ref_residual", "ref_shared"]          # primary: induced shift + 15 refs
         + [f"content_{n}" for n in SUB_ORDER])              # control: topic directions

SUMMARY = []
for lname, per_layer in GAINS.items():
    print(f"\n===== lens: {lname} =====")
    for bname, rng_ in BANDS.items():
        Ls = [L for L in rng_ if L in per_layer]
        if not Ls: continue
        chance = np.mean([per_layer[L]["chance"] for L in Ls])
        print(f"\n  {bname}  (chance capture = {chance:.2f}, chance gain = 1.00x)")
        print(f"  {'sub-trait':16s} {'prediction':28s} {'capture':>8s} {'gain':>8s}")
        for n in ORDER:
            if n not in per_layer[Ls[0]]: continue
            cap  = np.mean([per_layer[L][n]["capture"]  for L in Ls if n in per_layer[L]])
            gain = np.mean([per_layer[L][n]["gain_rel"] for L in Ls if n in per_layer[L]])
            if n.startswith("content_"):
                pred = "topic control"
            elif n.startswith("ref_"):
                pred = "15 reference"
            else:
                pred = SUBTRAITS[n][2]
            print(f"  {n:16s} {pred:28s} {cap:8.2f} {gain:7.2f}x")
            SUMMARY.append({"lens": lname, "band": bname, "subtrait": n,
                            "prediction": pred, "capture": float(cap), "gain_rel": float(gain)})

## 12. Save everything
`components_v1/` on Drive: the four experiment tables (JSON), the sub-trait gain summary (CSV),
and the sub-trait direction vectors (NPZ, dark + base) for the lens-lab `/api/dirwords` UI.

In [ ]:
import json as _json

with open(OUT / "exp1_dark_binary.json", "w") as f:
    _json.dump({"dark": EXP1, "base_control": EXP1B if "EXP1B" in dir() else None}, f, indent=2)
with open(OUT / "exp2_dep_binary.json", "w") as f:
    _json.dump({"all_internalizing": EXP2, "core_depression": EXP2_CORE}, f, indent=2)
with open(OUT / "exp3_probe_wanting.json", "w") as f:
    _json.dump({"cosines": EXP3A, "willingness": EXP3B}, f, indent=2)

with open(OUT / "exp4_subtrait_gains.csv", "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=["lens", "band", "subtrait", "prediction", "capture", "gain_rel"])
    wr.writeheader(); wr.writerows(SUMMARY)

np.savez_compressed(OUT / "subtrait_dirs_shift.npz", layers=np.array(ACT_LAYERS),
                    **{f"{n}_L{L}": v for L in SUBVEC for n, v in SUBVEC[L].items()})
np.savez_compressed(OUT / "subtrait_dirs_content.npz", layers=np.array(ACT_LAYERS),
                    **{f"{n}_L{L}": v for L in CONTENT for n, v in CONTENT[L].items()})
print("saved ->", OUT)
print(sorted(p.name for p in OUT.iterdir()))

---
# Done — full retrain analysis complete

Fresh numbers for the paper: the 2×2 capture/gain (15), and all four experiment tables in
`components_v1/` (16). Everything previously quoted was old-organism — replace it.
